# Developing data simulation

The objective of this notebook is helping to develop step by step the simulation from `R` into `python`

The first objective will be the creation of a simulation possibility based only on continuous features. No categorical will be considered.

In [1]:
import torch

In [2]:
means = torch.tensor([3.5,-3.5], dtype=torch.float64)
covs = torch.tensor([[1,-0.5],[-0.5,1]], dtype=torch.float64)
mvn = torch.distributions.MultivariateNormal(means, covariance_matrix=covs)

In a 2 dimensional multivariate distribution, we have a vector $X$ of two RV
$$
X = \begin{pmatrix} X_1 \\ X_2 \end{pmatrix}
$$
Mean and VCOV matrix are given by
$$
\mu = \begin{pmatrix} 3.5 \\ -3.5 \end{pmatrix}, \; 
\Sigma = \begin{pmatrix}
1 & -\frac{1}{2} \\
-\frac{1}{2} & 1
\end{pmatrix}
$$

This means when sampling we will only get positive numbers for $X_1$ realizations and negative numbers for $X_2$ realizations

In [3]:
sample_2d = mvn.sample((3,))
print(sample_2d.shape)
sample_2d

torch.Size([3, 2])


tensor([[ 4.0739, -2.5579],
        [ 3.5667, -2.2065],
        [ 4.5279, -4.0313]], dtype=torch.float64)

So in a sample size of `(n,)`, the shape will be `[n, k]` with `k` the number of random variables composing $X$.

With $x^{(r)}_j$ the $j$-th sampled realization of $X_r$, the output looks like
$$
\texttt{sample} = \begin{pmatrix}
    x^{(1)}_1 & \cdots & x^{(k)}_1 \\
    \vdots & \ddots & \vdots \\
    x^{(1)}_n & \cdots & x^{(k)}_n
\end{pmatrix} \in \mathbb{R}^{n \times k}
$$

## Notes on the R-Algorithm

### Calls:

First call is made for the init_population, by

```r
res <- genCreditData(
  #################################### DIMENSIONALITY
  n                = init_sample,  # - sample size = 100
  bad_ratio        = bad_ratio,    # - BAD ratio (if all D = 0) = 0.7
  k_con            = num_feats,    # - no. continuous features = 2
  k_cat            = 0,            # - no. categorical features
  k_bin            = 0,            # - no. binary features
  k_noise          = num_noise,    # - no. white-noise features = 0
  #################################### CONTINUOUS FEATURES
  con_nonlinear    = 0.0,       # - share of nonlinear transformations
  con_mean_bad_dif = mean_dif,  # - mean difference between classes = c(2, 1)
  con_var_bad_dif  = var_dif,   # - share of var/covar difference between classes = 0.5
  con_noise_var    = noise_var, # - variance of noise = 0
  covars           = covars,    # - variance-covariance matrices = list(matrix(c(1,  0.2,  0.2, 1), nrow = 2), matrix(c(1, -0.2, -0.2, 1), nrow = 2))
  #################################### MIXTURE OF GAUSSIANS
  mixture      = mixture,       # - mixture of two Gaussians = FALSE
  mix_mean_dif = mix_mean_dif,  # - mean difference between components  = 5
  mix_var_dif  = mix_var_dif,   # - share of var/covar difference between components = 0
  #################################### OTHER PARAMETERS
  seed             = seed,   # - random seed
  verbose          = F,      # - displaying feedback
  encode_factors   = F)      # - encoding of categorical features
```

`matrix(c(1,  0.2,  0.2, 1)` liefert
$$
\begin{pmatrix}
    1 & 0.2 \\
    0.2 & 1
\end{pmatrix}
$$

The next call is done this way:
```r
new_applicants <- genCreditData(n = sample_size, replicate = res, seed = seed + g)$data
```

Which is a quick way to replicate the arguments of the last call, based on this object (all names are set as objects - which are "`names`" in R)
```r
list(n                = n,
     k_cat            = k_cat,
     k_bin            = k_bin,
     k_noise          = k_noise,
     bad_ratio        = bad_ratio,
     con_mean_bad_dif = con_mean_bad_dif,
     con_var_bad_dif  = con_var_bad_dif,
     con_nonlinear    = con_nonlinear,
     con_noise_var    = con_noise_var,
     mixture          = mixture,
     mix_mean_dif     = mix_mean_dif,
     mix_var_dif      = mix_var_dif,
     cat_levels       = cat_levels,
     cat_var_share    = cat_var_share,
     cat_nonlinear    = cat_nonlinear,
     cat_noise_var    = cat_noise_var,
     bin_prob         = bin_prob,
     bin_mean_bad_dif = bin_mean_bad_dif,
     bin_bad_ratio    = bin_bad_ratio,
     bin_mean_con_dif = bin_mean_con_dif,
     bin_var_bad_dif  = bin_var_bad_dif,
     bin_noise_var    = bin_noise_var,
     encode_factors   = encode_factors,
     verbose          = verbose,
     seed             = seed)
```

however n and seed are not "taken over" - the rest they are

### Relevant parts of the algorithm

1. Set `combo_bad_ratio <- 0.7` and `combo_count <- n`
2. Compute "good" and "bad" sample sizes, as $n \cdot 0.7$ und $n \cdot (1-0.7)$ correspondingly. Ties are solved by 50/50 chance of doing good = n - bad or bad = n - good.
3. Set `mu_1 = c(0,0)` and therefore `mu_2 = c(1,2)=con_mean_bad_dif` (see line 249)

In [239]:
from typing import List, Optional, Tuple, Union
import torch
def random_vcov_matrix(
        k: int,
        generator: Optional[torch.Generator] = None,
        var_range: Tuple[float, float] = (0.0, 1.0),
        prefer_normal_base_sampling: bool = True,
        device: torch.device = torch.get_default_device(),
        dtype: torch.dtype = torch.get_default_dtype(),
        eps: float = 1e-6
    ) -> torch.Tensor:
    """Generate a random positive definite covariance matrix.

    The covariance matrix is produced by:

    1. Generating a base sampling of a matrix :math:`A` (normal or uniform).
    2. Creating a correlation matrix via cosine similarity between the row vectors
       of the base sampling:

       .. math::

          C = (c_{i,j}) = \\left( \\frac{\\langle A_i, A_j \\rangle}{\\|A_i\\| \\|A_j\\|} \\right)

    3. Sampling a variance vector uniformly in ``var_range``, which is used to
       rescale the correlation matrix.
    4. Ensuring positive definiteness via addition of a small diagonal
       perturbation ``eps``.
    5. Make sure exact symmetry, so that rounding point instability does not
       lead to unsymmetric results. 

    Args:
        k (int): Dimension of the covariance matrix.
        generator (torch.Generator, optional): Random number generator for reproducibility.
        var_range (Tuple[float, float], optional): Range for diagonal variances. Defaults to (0.0, 1.0).
        prefer_normal_base_sampling (bool, optional): If True, use normal distribution for base sampling.
            If False, use uniform distribution. Defaults to True.
        device (torch.device, optional): Device on which to allocate the tensor.
            Defaults to ``torch.get_default_device()``.
        dtype (torch.dtype, optional): Data type of the returned tensor.
            Defaults to ``torch.get_default_dtype()``.
        eps (float, optional): Small positive value added to the diagonal to ensure positive definiteness.
            Defaults to 1e-6.

    Returns:
        torch.Tensor: A symmetric, positive definite covariance matrix of shape ``(k, k)``.

    Raises:
        ValueError: If ``var_range`` is not a valid (min, max) tuple.

    Example:
        >>> g = torch.Generator().manual_seed(42)
        >>> cov = random_vcov_matrix(4, generator=g)
        >>> cov.shape
        torch.Size([4, 4])
    """

    # Step 1: Generate base sampling
    if prefer_normal_base_sampling:
        A = torch.randn((k, k), generator=generator, dtype=dtype, device=device) # random normal matrix, sparser correlations for high k
    else:
        A = 2*torch.rand((k,k), generator=generator, dtype = dtype, device=device) - 1 # random uniform matrix, correlations closer to 0, the higher k

    # Step 2: Define correlation matrix from base sampling
    Q = A @ A.T # Make sure of symmetry while using full randomness
    D = torch.sqrt(torch.diag(Q)) # Help vector for normalization
    corr_mat = Q / torch.outer(D, D) # corr_mat[i, j] = cosine_similarity(A[i], A[j]), so range [-1, 1] guaranteed

    # Step 3: Rescale corr_mat with sampled variances
    ## Variance sampling from uniform distribution
    variances = torch.rand(k, generator=generator, dtype = dtype, device=device) * (var_range[1] - var_range[0]) + var_range[0]
    ## Rescaling via outer prouct of standard deviations
    stds = torch.sqrt(variances)
    norm_factors_pearson_corr = torch.outer(stds, stds) # guaranteed to be symmetric, denominators of pearson correlation
    vcov = corr_mat * norm_factors_pearson_corr

    # Step 4: Avoid semi positive definitness of the matrix
    vcov = vcov + eps * torch.eye(k, device=device, dtype=dtype)

    # Step 5: Ensure **exact** symmetry without compromising randomness
    i, j = torch.tril_indices(k, k, offset=-1)
    vcov[i, j] = vcov[j, i]

    
    return vcov

def eigen_decomp_proj_to_pd(
    mat: torch.Tensor,
    eps: float = 1e-6,
    ensure_symmetry: bool = False
) -> torch.Tensor:
    """Project a matrix onto the positive definite (PD) cone via eigen-decomposition.

    The procedure ensures the output is symmetric and positive semidefinite by:
    
    1. Optionally symmetrizing the input matrix.
    2. Performing eigen-decomposition.
    3. Clipping eigenvalues below ``eps`` to enforce non-negativity.
    4. Reconstructing the matrix from clipped eigenvalues and eigenvectors.
    5. Symmetrizing the result again to avoid numerical drift.

    Args:
        mat (torch.Tensor): Input square matrix of shape ``(k, k)``.
        eps (float, optional): Minimum eigenvalue threshold to enforce positive definiteness.
            Defaults to ``1e-6``.
        ensure_symmetry (bool, optional): If True, symmetrize the input before decomposition.
            Defaults to False.

    Returns:
        torch.Tensor: Symmetric positive semidefinite matrix of shape ``(k, k)``.

    Example:
        >>> M = torch.tensor([[1.0, 2.0], [2.0, -3.0]])
        >>> M_psd = eigen_decomp_proj_to_pd(M)
        >>> torch.linalg.eigvalsh(M_psd)
        tensor([1.0133e-06, 1.8284e+00])
    """
    # Ensure symmetry
    if ensure_symmetry:
        mat = (mat + mat.T) / 2
    
    # Eigen-decomposition
    eigvals, eigvecs = torch.linalg.eigh(mat)
    
    # Clip eigenvalues to non-negative
    eigvals_clipped = torch.clamp(eigvals, min=eps)
    
    # Reconstruct
    mat_psd = eigvecs @ torch.diag(eigvals_clipped) @ eigvecs.T
    
    # Ensure symmetry again
    return (mat_psd + mat_psd.T) / 2

def generate_sigma_bad_and_good(
    k: int,
    proportion_var_dif: float,
    generator: torch.Generator,
    var_range: Tuple[float, float] = (0.0, 1.0),
    eps: float = 1e-6,
    device: torch.device = torch.device("cpu"),
    dtype: torch.dtype = torch.float64
) -> Tuple[torch.Tensor, torch.Tensor]:
    """Generate a pair of covariance matrices: one 'good' baseline and one 'bad' perturbed version.

    The construction proceeds as follows:

    1. Generate two baseline covariance matrices using ``random_vcov_matrix``.
    2. Sample a random mask over the upper-triangular entries (including diagonal).
    3. Copy selected entries from the 'good' matrix into the 'bad' matrix, leaving
       others perturbed.
    4. Reflect the upper-triangular entries to the lower-triangular part to ensure symmetry.
    5. Project the 'bad' matrix onto the positive definite cone using
       :func:`eigen_decomp_proj_to_pd`.

    Args:
        k (int): Dimension of the covariance matrices.
        proportion_var_dif (float): Probability of keeping an entry different between
            the 'bad' and 'good' matrices.
        generator (torch.Generator): Random number generator for reproducibility.
        var_range (Tuple[float, float], optional): Range for diagonal variances.
            Defaults to (0.0, 1.0).
        eps (float, optional): Small diagonal perturbation to ensure positive definiteness.
            Defaults to ``1e-6``.
        device (torch.device, optional): Device for tensor allocation. Defaults to CPU.
        dtype (torch.dtype, optional): Data type of the returned tensors. Defaults to ``torch.float64``.

    Returns:
        Tuple[torch.Tensor, torch.Tensor]:
            - ``sigma_bad``: Perturbed covariance matrix of shape ``(k, k)``, projected to PSD.
            - ``sigma_good``: Baseline covariance matrix of shape ``(k, k)``.

    Example:
        >>> g = torch.Generator().manual_seed(123)
        >>> sigma_bad, sigma_good = generate_sigma_bad_and_good(3, 0.5, generator=g)
        >>> sigma_bad.shape, sigma_good.shape
        (torch.Size([3, 3]), torch.Size([3, 3]))
    """
    # Step 1: Generate baseline matrices
    sigma_bad = random_vcov_matrix(k, generator=generator, var_range=var_range, device=device, dtype=dtype, eps=eps)
    sigma_good = random_vcov_matrix(k, generator=generator, var_range=var_range, device=device, dtype=dtype, eps=eps)

    # Step 2: Random mask for off-diagonal entries
    count_possible_changes = (k**2 + k) // 2 #Count diagonal entries + upper triangle
    index_change_vars = ~torch.bernoulli(torch.full((count_possible_changes,), proportion_var_dif, device=device), generator=generator).bool()

    triu_indices = torch.triu_indices(k, k, offset=0)
    indices_to_copy_sigma_bad = (triu_indices[0][index_change_vars], triu_indices[1][index_change_vars])

    sigma_good[indices_to_copy_sigma_bad] = sigma_bad[indices_to_copy_sigma_bad]
    i, j = torch.tril_indices(k, k, offset=-1)
    sigma_good[i, j] = sigma_good[j, i] # ensure symmetry
    
    sigma_good = eigen_decomp_proj_to_pd(sigma_good, eps=eps)

    return sigma_bad, sigma_good

def mvn_random_sample(
        mean : torch.Tensor, 
        cov_chol_decomp : Optional[torch.Tensor],
        n : int, 
        rng : Optional[torch.Generator] = None, 
        args_checks : bool = True
    ):
    """Generate n-vectors sampled of a multivariate normal (MVN) distribution with parameters
    mean and cov. Based on the implementation of (r)sample from 
    torch.distributions.MultivariateNormal according to torch version 2.9.1. It uses
    cholesky-decomposition method.

    Args:
        mean (torch.Tensor): Location parameter of a MVN. Shape ``(k,)`` or ``(b, k)`` or ``(1,5)``.
        cov_chol_decomp (torch.Tensor): Variance-Covariance matrix of MVN after cholesky decomposition. 
            Shape ``(k,k)``` or ``(b, k, k)`` or ``(1, k, k)``, ``cov.dim()==mean.dim()+1`` should hold.
        n (int): Count of vectors to be sampled (per batch).
        rng (Optional[torch.Generator]): If passed, sampling is done using this
            generator.
        args_checks (bool): If true, it will be checked whether the shapes of mean and
            cov are as expected, whether symmetry (w. r. t. to the last wo dims for each beach)
            is given within the range of ``symmetry_rtol_atol`` for ``cov`` and type checks
            are done for ``n`` and ``rng``.`
        symmetry_rtol_atol (Tuple[float,float]): Corresponds to the (rtol, a_tol) parameters
            of ``torch.allclose``, passed as ``*args``, so ordering is important. Ignored if
            ``not args_checks``.
    Returns:
        torch.Tensor:
            A tensor of shape ``(n, k)`` or ``(n, b, k)`` containing the ``n`` sampled vectors (for each batch).

    Example:
        >>> count_covariates = 5
        >>> device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        >>> torch.set_default_dtype(torch.float32)
        >>> rng = torch.Generator(device)
        >>> mu = torch.zeros(count_covariates)
        >>> sigma_bad, _ = generate_sigma_bad_and_good(k = count_covariates, proportion_var_dif=1.0, generator = rng, device=device, dtype= torch.get_default_dtype())
        >>> sample = mvn_random_sample(mean=mu, cov=sigma_bad, n=100, rng=rng)
        >>> sample.shape
        torch.Size([100, 5])
    """
    if args_checks:
        #shape checks
        assert (mean.dim() in [1, 2, 3]) and (cov_chol_decomp.dim()==mean.dim()+1), "mean must be a single vector (rank 1 tensor) and cov a matrix (rank 2 tensor)"
        assert (mean.size(-1) == cov_chol_decomp.size(-1)) and (cov_chol_decomp.size(-1) == cov_chol_decomp.size(-2)), "mean must have shape [k] and cov shape [k, k]"
        #ensure n is an int
        n = int(n)
        assert isinstance(rng, torch.Generator) or rng is None, "rng needs to be None or a rng"

    shape = torch.Size([n]) + mean.shape
    
    eps = torch.empty(shape, dtype=mean.dtype, device = mean.device).normal_(generator=rng)
    cov_chol_decomp = torch.linalg.cholesky(cov) # occurs over the last dimension

    deviations = torch.matmul(cov_chol_decomp, eps.unsqueeze(-1)).squeeze(-1) # apply decomp to each sampled vector


    return mean + deviations


In [ ]:
from simulation import GaussianMixture


GaussianMixture.generate_good_bad_mixtures(
    2,
    1,
    
)

In [ ]:
def _mix_var_dif_as_expected(mix_var_dif, m, k):
    if isinstance(mix_var_dif, float):
        return True
    if not isinstance(mix_var_dif, torch.Tensor):
        return False
    
    dim_rank = mix_var_dif.dim()
    if dim_rank == 0 or (mix_var_dif.numel() == 1):
        return True
    
    dim_rank = mix_var_dif.dim()

    if dim_rank == 1:
        return mix_var_dif.size(-1) in (m-1, 1)
    
    if dim_rank == 2:
        return mix_var_dif.shape in (torch.Size([m-1, 1]), torch.Size([k,k]))
    
    if dim_rank > 3:
        return False
    
    last_dims_ok = mix_var_dif.size(1) == mix_var_dif.size(2) == k
    return last_dims_ok and (mix_var_dif.size(0) in (m-1, 1))


def _adapt_mix_var_dif(mix_var_dif, m, k, security_check : bool = True, dtype : torch.dtype = torch.get_default_dtype()):
    if security_check:
        assert _mix_var_dif_as_expected(mix_var_dif, m, k)

    is_float = isinstance(mix_var_dif, float)
    is_single_element_tensor = not is_float and (mix_var_dif.numel() == 1)
    is_singleton = is_float or (mix_var_dif.dim() == 0) or is_single_element_tensor
    if is_singleton:
        if is_float:
            mix_var_dif = torch.tensor(mix_var_dif, dtype = dtype)
        if is_single_element_tensor:
            mix_var_dif = mix_var_dif.flatten()[0]
        return mix_var_dif#.expand(m-1).unsqueeze(-1).unsqueeze(-1)
    
    if mix_var_dif.dim() == 1:
        return mix_var_dif.unsqueeze(-1).unsqueeze(-1)
        
    if mix_var_dif.dim() == 2:
        if mix_var_dif.size(0) == k:
            return mix_var_dif.unsqueeze(0)
        if mix_var_dif.size(0) == (m-1):
            return mix_var_dif.unsqueeze(-1)
    
    if mix_var_dif.dim() == 3:
        return mix_var_dif
    


In [5]:
import torch

rng = torch.Generator()
rng.initial_seed()

67280421310721

In [35]:
%run gaussian_mixture.py

Step 1: checking functionality of eigen_decomp_proj_to_pd
	Shape mantained: True
	Result is positive definite: True

Step 2: Checking if matrices can be constructed with random vcovs
	all_covs.size is as expected: True
	Generated covs are symmetric: True
	Generated covs are positive definite: True


Step 3: Attempts to create and sample with different shapes of params the GaussianMixture


Attempt 1: no weights, unbatched params
Params shapes:
  mean.size = torch.Size([5])
  cov.size = torch.Size([5, 5]) 

	Expected size: torch.Size([100, 5])
	Realized size: torch.Size([100, 5])
	Expectation realized: True 


Attempt 2: no weights, batched params
Params shapes:
  mean.size = torch.Size([4, 5])
  cov.size = torch.Size([4, 5, 5]) 

	Expected size: torch.Size([4, 100, 5])
	Realized size: torch.Size([4, 100, 5])
	Expectation realized: True 


Attempt 3: weigths, batched params, weight based deterministic mvn sampling
Params shapes:
  mean.size = torch.Size([4, 4, 5])
  cov.size = torch.Siz

In [2]:
torch.get_default_dtype()

torch.float64

In [1]:
import torch

from credit_data_simulation import CreditDataGenerator

torch.set_default_dtype(torch.float64)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

initial_seed = 1807

#Acceptance loop parametrization
init_sample = 100
sample_size = 100
holdout_sample = 3000
num_gens = 300
top_percent = 0.2

determinstic_mixture_weights = True

data_gen = CreditDataGenerator.init_with_internal_logic(
    count_covariates=2,
    mean_bad_diff=torch.tensor([1.0,2.0]),
    covars = {
        "bad" : torch.tensor([[1.0, 0.2], [0.2,1.0]]),
        "good" : torch.tensor([[1.0,-0.2], [-0.2,1.0]])
    },
    iid = False,
    mixture_weights=None,
    bad_ratio = 0.5,
    noise_var=0.0,
    device = device,
    seed_credit_data_gen=initial_seed,
    #dtype=torch.get_default_dtype()
)

X, y = data_gen.sample(100)



array([[-0.1013214 ,  1.67434285],
       [ 0.04440619, -0.36218245],
       [ 0.67122243,  0.39064223],
       [-1.93769365, -0.31725522],
       [-0.18209041,  0.20263129],
       [-1.56341658,  0.54088715],
       [ 0.63735175,  1.60117883],
       [-0.35225355, -1.65931021],
       [-0.95950671, -0.95082175],
       [ 0.58922719, -0.08849382],
       [-1.18607651,  0.03257276],
       [-0.61599367,  1.19022367],
       [-0.70101087, -1.46882702],
       [ 1.20036211,  0.19842806],
       [-0.01130603,  1.21973596],
       [ 1.05667895,  1.76607951],
       [-0.79017514, -0.02441563],
       [ 0.55566907,  0.49050609],
       [-0.88727315, -0.98250332],
       [ 0.51095276,  0.75622198],
       [ 1.22630684,  0.92389252],
       [-1.85902672, -0.83809114],
       [ 0.36949057,  1.90747146],
       [-0.86124383, -0.13911259],
       [-2.61844079, -0.3469208 ],
       [-0.29274318, -0.02786451],
       [-0.91428163,  0.60259028],
       [-0.53741127, -0.15717889],
       [ 0.62049226,

In [52]:
from sklearn.datasets import load_iris

from sklearn.linear_model import LogisticRegression
import numpy as np

iris_frame = load_iris(as_frame=True)
desired = "versicolor"
idx_desired = np.where(iris_frame["target_names"] == desired)[0][0]

y = (iris_frame["target"] == idx_desired).astype(int).values
X = iris_frame["data"].values
X = np.column_stack([torch.ones((X.shape[0],1)), X])

clf = LogisticRegression(random_state=0, solver='newton-cholesky').fit(X, y)
clf.predict_proba(X)

array([[0.88676942, 0.11323058],
       [0.72271561, 0.27728439],
       [0.80494734, 0.19505266],
       [0.74027084, 0.25972916],
       [0.90491697, 0.09508303],
       [0.95299568, 0.04700432],
       [0.86796902, 0.13203098],
       [0.85292081, 0.14707919],
       [0.65823805, 0.34176195],
       [0.7258542 , 0.2741458 ],
       [0.92186886, 0.07813114],
       [0.8391836 , 0.1608164 ],
       [0.69265486, 0.30734514],
       [0.71746209, 0.28253791],
       [0.96730637, 0.03269363],
       [0.98616774, 0.01383226],
       [0.96401623, 0.03598377],
       [0.89895249, 0.10104751],
       [0.93835975, 0.06164025],
       [0.94017075, 0.05982925],
       [0.84423533, 0.15576467],
       [0.9351835 , 0.0648165 ],
       [0.92129241, 0.07870759],
       [0.85886186, 0.14113814],
       [0.80894417, 0.19105583],
       [0.69774202, 0.30225798],
       [0.87468021, 0.12531979],
       [0.88146975, 0.11853025],
       [0.86567239, 0.13432761],
       [0.77003438, 0.22996562],
       [0.

In [ ]:
import numpy as np
import statsmodels.api as sm
from statsmodels.genmod import families # This module holds the GLM families


def fit_and_predict_classic_logistic(X : np.array, y : np.array, add_intercept : bool = True):
    if add_intercept:
        X = np.column_stack([np.ones((X.shape[0],1), X.dtype), X])

    model = sm.GLM(
        endog=y,
        exog=X_with_const,
        family=binomial_family # This specifies the logistic regression setup
    )
    fitted_model = model.fit()
    preds = fitted_model.predict()
    
    return preds
    

# --- 1. Prepare Data (Using the Iris data structure from your first prompt) ---

# Assume you still have X (features) and y (binary target) from the first cell
# We will regenerate them here for completeness:
from sklearn.datasets import load_iris
iris_frame = load_iris(as_frame=True)
desired = "versicolor"
idx_desired = np.where(iris_frame["target_names"] == desired)[0][0]
y = (iris_frame["target"] == idx_desired).astype(int).values
X_data = iris_frame["data"].values

# --- 2. CRITICAL STEP: Add the Intercept (Constant) ---
# The GLM function, like OLS, requires a constant/intercept term be added manually
X_with_const = sm.add_constant(X_data, prepend=False) # 'prepend=False' puts const at the end

# --- 3. Define and Fit the GLM (Logistic Regression) ---
# Define the family and link: Binomial(link=logit)
binomial_family = families.Binomial()

# Initialize the GLM: GLM(endog=y, exog=X, family=...)
model = sm.GLM(
    endog=y,
    exog=X_with_const,
    family=binomial_family # This specifies the logistic regression setup
)
results = model.fit()

# --- 4. View the Summary ---
long_predict= results.predict()
short_predict=fit_and_predict_classic_logistic(X_data, y)
print(results.predict())

print(fit_and_predict_classic_logistic(X_data, y) )

[0.08491322 0.28291787 0.17198267 0.26801458 0.0670752  0.02340324
 0.09510061 0.1254467  0.37105382 0.30992032 0.05320423 0.14661551
 0.34804048 0.28923994 0.01462699 0.00421107 0.01397163 0.06566779
 0.03742349 0.03347754 0.14464349 0.03353656 0.04479519 0.09470572
 0.20305583 0.33362371 0.08579165 0.09359066 0.10695126 0.23549347
 0.28446379 0.06941931 0.02482739 0.01173785 0.25382289 0.14472203
 0.06869591 0.08865733 0.28115903 0.12277954 0.05940959 0.67183757
 0.18271903 0.03911073 0.04248433 0.23445354 0.04956519 0.19533023
 0.05445382 0.14263928 0.26823691 0.19830289 0.32860539 0.77550208
 0.45723491 0.61042772 0.15880291 0.73519738 0.51998858 0.4465612
 0.91513226 0.2480506  0.90292239 0.51404762 0.27125004 0.26029894
 0.34494454 0.73346562 0.80979893 0.74651279 0.15268869 0.42412709
 0.75225128 0.70920599 0.43412893 0.3229554  0.57333639 0.30787435
 0.38702802 0.62908278 0.77780622 0.80208828 0.54831874 0.64790501
 0.35611535 0.10565235 0.28329881 0.82761478 0.35180266 0.66381

In [4]:
y

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [ ]:
from typing import Optional, List, Union, Dict, Any

from simulation import GaussianMixture

torch.set_default_dtype(torch.float64)

good_mixture, bad_mixture = GaussianMixture.generate_good_bad_mixtures(
    count_covariates=2,
    mean_bad_diff=torch.tensor([2.0,1.0]),
    covars = {
        "bad" : torch.tensor([[1.0, 0.2], [0.2,1.0]]),
        "good" : torch.tensor([[1.0,-0.2], [-0.2,1.0]])
    },
    iid = False,
    mixture_weights=None
)

def gen_credit_data(
        bad_mixture : GaussianMixture,
        good_mixture : GaussianMixture,
        n : int = 1000,
        bad_ratio : float = 0.5,
        seed   : Optional[int] = 1807,
        share_rng : bool = None
    ):
    """
    Copy of genCreditData - nur für kontinuierliche Varianten
    mean_bad_dif 
        if torch.Tensor then it has to have the shape (count_covariates,)
    """
    # 0. Process "None" logic path and ensure everything is well set
    ## ensure device is defined
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ## Set rng logic
    rng = torch.Generator(device)
    if seed is not None:
        rng = rng.manual_seed(seed)


    #
    #combo_bad_ratio = bad_ratio # Not too much sense but helps


    # compute sample sizes
    n_bad = round(bad_ratio * n)
    n_good = round((1-bad_ratio) * n)
    if (n_bad + n_good) != n:
        adapt_n_bad = torch.randint(low=0,high=2,size=(1,),generator=rng).to(bool).item()
        if adapt_n_bad:
            n_bad = n - n_good
        else:
            n_good = n - n_bad
    
    kwargs_for_generated_tensors = {"dtype" : dtype, "device" : device}
    if calculated_vars is None:
        mu_bad = torch.zeros(count_covariates, **kwargs_for_generated_tensors)
        mu_good = mu_bad + mean_bad_dif
        if mixture:
            mu_bad_mix = mu_bad + mix_mean_dif
            mu_good_mix = mu_good + mix_mean_dif

        if iid:
            # Identity matrices
            sigma_bad = torch.eye(count_covariates, **kwargs_for_generated_tensors)
            sigma_good = torch.eye(count_covariates, **kwargs_for_generated_tensors)
            if mixture:
                sigma_bad_mix = sigma_bad.detach().clone()
                sgima_good_mix = sigma_good.detach().clone()
        else:
            if covars is not None:
                sigma_bad, sigma_good = covars["bad"], covars["good"]
            else:
                sigma_bad, sigma_good = generate_sigma_bad_and_good(k = count_covariates, proportion_var_dif=con_var_bad_dif, generator = rng, device=device, dtype= dtype)

            if mixture:
                sigma_bad_mix = sigma_bad + mix_var_dif
                sigma_good_mix = sigma_good + mix_var_dif
        
    else:
        mu_bad = calculated_vars["mu_bad"]
        mu_good = calculated_vars["mu_good"]

        if mixture:
            mu_bad_mix = calculated_vars["mu_bad_mix"]
            mu_good_mix = calculated_vars["mu_good_mix"]

        sigma_bad = calculated_vars["sigma_bad"]
        sigma_good = calculated_vars["sigma_good"]
        if mixture:
            sigma_bad_mix = calculated_vars["sigma_bad_mix"]
            sigma_good_mix = calculated_vars["sigma_good_mix"]

    

    # Generation logic
    if seed is not None:
        rng = rng.manual_seed(seed)
        
    X_bad = mvn_random_sample()
    

In [9]:
torch.empty((0,5)).normal_()

tensor([], size=(0, 5))

In [7]:
import pandas as pd

adaptations = pd.DataFrame(
    [
        ("combo_bad_ratio", "bad_ratio"),
        ("combo_count", "n"),
        ("combo_n1", "n_bad"),
        ("combo_n2", "n_good"),
        ("k_con", "count_covariates"),
        ("con_mean_bad_dif", "mean_bad_dif")
    ],
    columns = ["Old", "New"]
)
adaptations

,Old,New
0,combo_bad_ratio,bad_ratio
1,combo_count,n
2,combo_n1,n_bad
3,combo_n2,n_good
4,k_con,count_covariates
5,con_mean_bad_dif,mean_bad_dif
